### TripleBarrierLabeler

In [3]:
import numpy as np 
from datetime import date, timedelta
from typing import Optional

from mps.core.config import cfg, msg 
from mps.core.types import Bar 
from mps.core.calendar import market_open_dt 
from mps.core.libs import logger
from mps.data.features import TripleBarrierLabeler 

In [2]:
TICKER = '005930'
BUY = cfg.data.dir2idx[cfg.str.buy]     # 0
HOLD = cfg.data.dir2idx[cfg.str.hold]   # 1
BASE_DATE = date(2025, 3, 4)
T0 = market_open_dt(BASE_DATE)
logger.debug(T0)

2026-06-22 17:15:05.982    DEBUG: 2025-03-04 09:00:00+09:00


In [4]:
def make_bar(
    open: float, high: float, low: float, 
    close: Optional[float] = None, 
    offset_min: int = 0
) -> Bar:
    return Bar(
        ticker=TICKER,
        timestamp=T0 + timedelta(minutes=offset_min),
        open=open, high=high, low=low, close=open if close is None else close,
        volume=1000,
        is_complete=True
    )
    
def neutral(offset_min: int = 0) -> Bar:
    """ 어떤 장벽도 건드리지 않는 평탄 봉 (진입가=100, 가정 시 high=101, low=99.5) """
    return make_bar(open=100, high=101, low=99.5, offset_min=offset_min)

logger.point("준비 완료")

2026-06-22 17:21:50.916    POINT: 준비 완료


#### label() ─ TripleBarrier 판정 검증

In [5]:
# 공통 labeler
labeler = TripleBarrierLabeler(time_horizon=3)
logger.debug(f"{labeler._take_profit}, {labeler._stop_loss}, {labeler.time_horizon}")

2026-06-22 17:24:46.363    DEBUG: 0.02, 0.01, 3


##### 익절 도달 → BUY (진입 봉 k=0에서 즉시)

In [7]:
# t=0: 진입가=bar[1].open=100, tp_line=102
# k=0: bar[1].high=103 >= 102 → tp 도달 → BUY

bars = [
    make_bar(open=100, high=100, low=100, offset_min=0),    # t=0 (신호봉)
    make_bar(open=100, high=103, low=100, offset_min=1),    # k=0: tp 도달
    neutral(offset_min=2),
    neutral(offset_min=3),
    neutral(offset_min=4),
]
labels = labeler.label(bars)
logger.debug(labels)

2026-06-22 17:30:40.962    POINT: eatures/_labeler.py[97]: 라벨링 대상 분봉 갯 수: 5개, 라벨링 결과: {'BUY': 1, 'HOLD': 4}
2026-06-22 17:30:40.962    DEBUG: [0 1 1 1 1]


In [ ]:
# TODO 9999-9999: 여기 체크